In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path
from IPython.display import display

# --- Configuration ---
# Define paths assuming this notebook is in the 'notebooks/' directory
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Load the CSV file that contains the final cluster labels
CLUSTERED_FILENAME = "rental_properties_clustered.csv"
CLUSTERED_FILEPATH = PROCESSED_DATA_DIR / CLUSTERED_FILENAME

# --- Load the Data ---
try:
    df_clustered = pd.read_csv(CLUSTERED_FILEPATH)
    print("Clustered data loaded successfully.")
    print("Shape:", df_clustered.shape)
    display(df_clustered.head())
except FileNotFoundError:
    print(f"ERROR: Clustered data file not found at {CLUSTERED_FILEPATH}")
    

Clustered data loaded successfully.
Shape: (2185, 29)


,price_eur_pm,barrio_encoded,adaptado_movilidad_reducida,aire_acondicionado,antiguedad,armarios_empotrados,ascensor,balcon,banos,calefaccion,...,sistema_seguridad,superficie_construida,terraza,trastero,vidrios_dobles,planta_numerica,amueblado_True,amueblado_nan,cluster,cluster_label
0,10000.0,3843.846154,0.0,1.0,2.623525,0.0,1.0,0.0,5.0,1.0,...,0.0,272.0,1.0,0.0,0.0,6.0,0.0,1.0,0,Piso Señorial Clásico
1,1450.0,2586.916667,0.0,0.0,0.747427,0.0,1.0,0.0,1.0,1.0,...,0.0,74.0,1.0,0.0,0.0,6.0,1.0,0.0,2,Propiedad Singular (Outlier)
2,2200.0,3843.846154,0.0,0.0,-0.213655,0.0,1.0,0.0,1.0,0.0,...,0.0,101.0,0.0,0.0,0.0,2.0,1.0,0.0,2,Propiedad Singular (Outlier)
3,1700.0,1932.200000,0.0,0.0,0.000000,0.0,1.0,0.0,1.0,1.0,...,0.0,65.0,1.0,0.0,0.0,6.0,1.0,0.0,2,Propiedad Singular (Outlier)
4,2650.0,3353.040000,0.0,1.0,0.000000,1.0,1.0,1.0,2.0,1.0,...,0.0,83.0,0.0,0.0,0.0,5.0,1.0,0.0,2,Propiedad Singular (Outlier)


In [2]:
print("--- Cluster Size Distribution ---")
cluster_counts = df_clustered['cluster'].value_counts().sort_index()
print(cluster_counts)

# Visualize the distribution
fig = px.bar(
    cluster_counts, 
    x=cluster_counts.index, 
    y=cluster_counts.values,
    title="Number of Properties per Cluster",
    labels={'x': 'Cluster ID', 'y': 'Number of Properties'}
)
fig.show()

--- Cluster Size Distribution ---
cluster
0     354
1     594
2    1040
3      30
4     167
Name: count, dtype: int64


In [4]:
print("--- Analysis of Numerical Features by Cluster ---")

# List of key numerical features to analyze
numerical_features = [
    'price_eur_pm',
    'superficie_construida',
    'habitaciones',
    'banos',
    'planta_numerica',
    'antiguedad', # This is now numerical (ordinal)
    'conservacion', # This is now numerical (ordinal)
    'barrio_encoded', # The target-encoded value
]

# Group by cluster and calculate the mean and median for each feature
# Using .agg() allows us to calculate multiple statistics at once
cluster_summary_numerical = df_clustered.groupby('cluster')[numerical_features].agg(['mean', 'median', 'count'])

# Display the summary table
# We'll round the values for easier reading
display(cluster_summary_numerical.round(2))

--- Analysis of Numerical Features by Cluster ---


price_eur_pm               superficie_construida               \
                mean  median count                  mean median count   
cluster                                                                 
0            3306.66  2300.0   354                155.57  120.0   354   
1            1929.65  1500.0   594                 84.86   71.0   594   
2            2344.77  2200.0  1040                 80.06   70.0  1040   
3            5661.67  5500.0    30                284.87  267.5    30   
4            3044.52  2700.0   167                128.89  111.0   167   

        habitaciones              banos  ... planta_numerica antiguedad  \
                mean median count  mean  ...           count       mean   
cluster                                  ...                              
0               2.86    3.0   354  2.45  ...             354       2.39   
1               2.16    2.0   594  1.48  ...             594       0.74   
2               1.86    2.0  1040  1.53  ...            1040       0.92   
3               4.13    4.0    30  4.00  ...              30       1.17   
4               2.36    2.0   167  1.97  ...             167       1.62   

                     conservacion              barrio_encoded                 
        median count         mean median count           mean   median count  
cluster                                                                       
0         2.15   354         1.36   1.18   354        2415.63  2258.45   354  
1         0.63   594         1.12   1.00   594        2279.17  2293.73   594  
2         0.61  1040         1.27   1.06  1040        2568.33  2428.85  1040  
3         1.00    30         1.69   2.00    30        3413.19  3480.11    30  
4         1.00   167         1.77   2.00   167        2701.84  2806.70   167  

[5 rows x 24 columns]

In [7]:
print("\n--- Analysis of Boolean (Amenity) Features by Cluster ---")

# 1. Start with the known, single boolean amenity columns
base_boolean_features = [
    'ascensor',
    'piscina',
    'garaje',
    'terraza',
    'trastero',
    'exterior',
    'acepta_mascotas' # This is specific to the rental dataset
]

# 2. Dynamically find the one-hot encoded columns for 'amueblado' that actually exist
amueblado_cols = [col for col in df_clustered.columns if col.startswith('amueblado_')]

# 3. Combine the lists to get the final list of features to analyze.
#    This ensures we only try to access columns that are present in the current DataFrame.
boolean_features_to_analyze = [col for col in base_boolean_features if col in df_clustered.columns] + amueblado_cols

print(f"Analyzing the following boolean/encoded features: {boolean_features_to_analyze}")

# Group by cluster and calculate the mean (percentage) for each boolean feature
cluster_summary_boolean = df_clustered.groupby('cluster')[boolean_features_to_analyze].mean()

# Display the summary table (values will be from 0.0 to 1.0)
# The .style call will now work on the columns that are confirmed to exist.
display((cluster_summary_boolean * 100).round(2).style.background_gradient(cmap='viridis').format("{:.2f}%"))


--- Analysis of Boolean (Amenity) Features by Cluster ---
Analyzing the following boolean/encoded features: ['ascensor', 'piscina', 'garaje', 'terraza', 'trastero', 'exterior', 'amueblado_True', 'amueblado_nan']


,ascensor,piscina,garaje,terraza,trastero,exterior,amueblado_True,amueblado_nan
cluster,,,,,,,,
0,79.94%,70.90%,72.60%,43.79%,59.04%,34.18%,19.21%,80.79%
1,58.42%,3.20%,10.10%,24.92%,5.22%,49.49%,0.00%,100.00%
2,77.69%,4.90%,6.73%,19.23%,4.13%,43.27%,100.00%,0.00%
3,66.67%,33.33%,53.33%,56.67%,23.33%,60.00%,30.00%,70.00%
4,92.22%,19.16%,41.32%,42.51%,22.75%,86.23%,63.47%,36.53%


In [11]:
print("\n--- Visualizing Price Distribution by Cluster ---")

fig = px.box(
    df_clustered,
    x='cluster',
    y='price_eur_pm',
    color='cluster',
    title='Distribution of Property Prices by Cluster',
    labels={'cluster': 'Cluster ID', 'price_eur': 'Price (€)'},
    points="all" # Show all underlying data points
)
fig.update_layout(xaxis_type='category') # Treat cluster IDs as categories
print("\n--- Visualizing Mean Feature Values by Cluster ---")

fig.show()


--- Visualizing Price Distribution by Cluster ---

--- Visualizing Mean Feature Values by Cluster ---


In [14]:
print("\n--- Visualizing Mean Feature Values by Cluster ---")

# Step 1: Define the features we want to analyze
features_to_plot = ['superficie_construida', 'banos', 'habitaciones']

# Step 2: Calculate the cluster_means DataFrame by grouping and aggregating
cluster_means = df_clustered.groupby('cluster')[features_to_plot].mean().reset_index()

# Step 3: Now that cluster_means exists, display it as a formatted table
print("\n--- Summary Table: Mean Feature Values by Cluster ---")
styled_means = cluster_means.style.format({
    'superficie_construida': '{:,.2f}', # Format with comma for thousands and 2 decimal places
    'banos': '{:.2f}',                  # Format to 2 decimal places
    'habitaciones': '{:.2f}'            # Format to 2 decimal places
}).set_caption("Average Feature Values per Cluster").hide(axis='index')
display(styled_means)

# Step 4: Proceed with creating the plot using the same cluster_means DataFrame
# We need to "melt" the DataFrame to make it suitable for a grouped bar chart
df_melted = cluster_means.melt(id_vars='cluster', value_vars=features_to_plot, var_name='Feature', value_name='Average Value')

fig = px.bar(
    df_melted,
    x='cluster',
    y='Average Value',
    color='Feature',
    barmode='group', # Group bars for the same cluster side-by-side
    title='Comparison of Average Feature Values Across Clusters',
    labels={'cluster': 'Cluster ID'}
)
fig.update_layout(xaxis_type='category')
fig.show()


--- Visualizing Mean Feature Values by Cluster ---

--- Summary Table: Mean Feature Values by Cluster ---


cluster,superficie_construida,banos,habitaciones
0,155.57,2.45,2.86
1,84.86,1.48,2.16
2,80.06,1.53,1.86
3,284.87,4.00,4.13
4,128.89,1.97,2.36


## Proposed Labels and Profiles for Each Cluster

### Cluster 2: Furnished City Apartments
This is the largest group (1040 properties), representing a very common segment of the Madrid rental market.

#### Defining Characteristics:
*   **Size**: Smallest average size (~80 m²) and fewest rooms/bathrooms.
*   **Price**: Lower-mid rental price (~€2,345/month).
*   **Amenities**: This cluster is 100% furnished (amueblado_True). It has a high rate of elevators (78%) but is low on other extras like pools or garages.
*   **Location**: Located in high-rent neighborhoods (barrio_encoded is high), indicating desirable, central locations.
*   **Persona**: This cluster perfectly captures the ready-to-rent, furnished apartments in sought-after city neighborhoods. They are ideal for young professionals, students, or anyone looking for a turnkey rental solution in a good area.

### Cluster 1: Standard & Budget-Friendly Flats
This is the second-largest group (594 properties) and represents the most affordable segment.

#### Defining Characteristics:

*   **Price**: Has the lowest average rent of all clusters (~€1,930/month).
*   **Location**: Located in neighborhoods with the lowest average rent (barrio_encoded is lowest).
*   **Amenities**: Has the fewest amenities across the board (lowest % of elevators, pools, garages). Critically, the furnished status is 100% unknown (amueblado_nan), which in the rental market often implies "unfurnished."
*   **Condition**: This is, on average, the oldest group of properties.
*   **Persona**: This cluster represents the entry-level or most budget-friendly rental properties. They are typically older, smaller, unfurnished apartments in less central or less expensive neighborhoods with minimal extras.

### Cluster 0: Family-Sized Modern Residences
This is a distinct group of 354 properties aimed at families or those needing more space and amenities.

#### Definining Characteristics:

*   **Price**: High-end rental price (~€3,300/month).
*   **Size**: Very spacious (~156 m²) with the second-most rooms (~2.9) and bathrooms (~2.5).
*   **Amenities**: This cluster is defined by its amenities suitable for residential complexes (urbanizaciones). It has a very high percentage of pools (71%) and garages (73%).
*   **Condition**: These are among the newest properties on average.
*   **Persona**: These are modern, large apartments or houses in residential areas, often on the outskirts of the central core, where complexes with pools and garages are more common. They are ideal for families looking for space and a higher quality of life with more amenities.

### Cluster 4: Exclusive Luxury Homes
This group of 167 properties represents the high-quality city living segment.

#### Defining Characteristics:

*   **Price**: High rental price (~€3,045/month), with a tighter and higher median than Cluster 0.
*   **Location**: Located in very high-rent neighborhoods.
*   **Amenities**: Has the highest rate of elevators (92%) and a very high rate of being exterior-facing (86%).
*   **Condition**: Has the best conservation status (highest score), indicating they are either new or recently renovated to a high standard.
*   **Size**: Medium-sized (~129 m²), smaller than the "family" cluster but larger than the "standard" ones.
*   **Persona**: This cluster represents premium apartments in prime, desirable city locations. They are not defined by having a pool, but by their excellent condition, location, and key apartment features like being outward-facing with an elevator. They are perfect for executives or anyone prioritizing location and quality over sheer size.

### Cluster 3: Premium Renovated Apartments (Prime Location)
This is your smallest group (30 properties), representing the absolute top-tier of the rental market.

#### Defining Characteristics:

*   **Price**: The highest by a large margin (~€5,660/month). The box plot shows its price range is far above all others.
*   **Size**: The largest by far (~285 m²) with the most rooms (~4.1) and bathrooms (~4.0).
*   **Location**: Located in the most expensive neighborhoods (barrio_encoded is highest).
*   **Amenities**: Well-equipped with high rates of pools, garages, and terraces.
*   **Persona**: This is the exclusive, high-standing luxury segment. These are likely large penthouses, duplexes, or standalone houses (chalets) for rent in the most prestigious areas of Madrid, catering to a very high-end market.

### Summary of Rental Property Clusters

| ID Cluster | Etiqueta | Características Clave                                                                        |
| :--------- | :-------------------------- | :------------------------------------------------------------------------------------------- |
| **0** | Residencial Familiar con Extras | Grande, precio elevado, nuevo, en urbanización con piscina y garaje.                         |
| **1** | Piso Básico y Económico     | El más barato, ubicación menos céntrica, pequeño, antiguo y con pocos extras. (Sin amueblar). |
| **2** | Apartamento Céntrico Amueblado | Tamaño mediano/pequeño, 100% amueblado, en barrios de alquiler alto, con ascensor.        |
| **3** | Vivienda de Lujo Exclusivo  | El más caro y grande, ubicación y calidades de lujo, con todos los extras.                 |
| **4** | Apartamento Premium (Ubicación Prime) | Precio elevado, ubicación prime, excelente estado (reformado/nuevo), exterior y con ascensor. |